# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [5]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

A_df = pd.read_csv("/workspaces/dx704-project-07/model-A.tsv", sep="\t")
B_df = pd.read_csv("/workspaces/dx704-project-07/model-B.tsv", sep="\t")
Q_df = pd.read_csv("/workspaces/dx704-project-07/cost-Q.tsv", sep="\t")
R_df = pd.read_csv("/workspaces/dx704-project-07/cost-R.tsv", sep="\t")
log_df = pd.read_csv("/workspaces/dx704-project-07/qc-log.tsv", sep="\t")
train_df = pd.read_csv("/workspaces/dx704-project-07/qc-train.tsv", sep="\t")

state_cols = ["x", "v", "a"]

A = A_df.set_index("index").loc[state_cols, state_cols].to_numpy(dtype=float)
B = B_df.set_index("index").loc[state_cols, ["u"]].to_numpy(dtype=float)
Q = Q_df.set_index("index").loc[state_cols, state_cols].to_numpy(dtype=float)
R = R_df.set_index("index").loc[["u"], ["u"]].to_numpy(dtype=float)

Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [6]:
# YOUR CHANGES HERE

P = solve_discrete_are(A, B, Q, R)
K_intended = np.linalg.inv(R + B.T @ P @ B) @ (B.T @ P @ A)

K_intended_df = pd.DataFrame([K_intended.flatten()], columns=state_cols)
K_intended_df.to_csv("control-K-intended.tsv", sep="\t", index=False)

print(K_intended_df)

         x         v         a
0  0.33446  1.304454  1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [7]:
# YOUR CHANGES HERE

X_log = log_df[state_cols].to_numpy(dtype=float)
u_check = -(X_log @ K_intended.flatten())

Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [8]:
# YOUR CHANGES HERE

qc_check = pd.DataFrame({
    "index": log_df["index"],
    "u_check": u_check
})

qc_check.to_csv("qc-check.tsv", sep="\t", index=False)

print(qc_check.head())

   index   u_check
0      0  1.672299
1      1 -1.152095
2      2 -1.235183
3      3 -0.288504
4      4  0.369202


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [9]:
# YOUR CHANGES HERE

X_log = log_df[state_cols].to_numpy(dtype=float)
u_log = log_df["u"].to_numpy(dtype=float)

K_actual_vec, _, _, _ = np.linalg.lstsq(-X_log, u_log, rcond=None)
K_actual = K_actual_vec.reshape(1, -1)

Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [10]:
# YOUR CHANGES HERE

K_actual_df = pd.DataFrame([K_actual.flatten()], columns=state_cols)
K_actual_df.to_csv("control-K-actual.tsv", sep="\t", index=False)

print(K_actual_df)

          x        v         a
0  0.340438  1.30012  1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [11]:
# YOUR CHANGES HERE

X_train = train_df[["x", "v", "a", "u"]].iloc[:-1].to_numpy(dtype=float)
Y_train = train_df[["x", "v", "a"]].iloc[1:].to_numpy(dtype=float)

coef, _, _, _ = np.linalg.lstsq(X_train, Y_train, rcond=None)

A_new = coef[:3, :].T
B_new = coef[3, :].reshape(-1, 1)

A_new_df = pd.DataFrame(A_new, columns=state_cols)
A_new_df.insert(0, "index", state_cols)

B_new_df = pd.DataFrame(B_new, columns=["u"])
B_new_df.insert(0, "index", state_cols)

Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [12]:
# YOUR CHANGES HERE

A_new_df.to_csv("model-A-new.tsv", sep="\t", index=False)
B_new_df.to_csv("model-B-new.tsv", sep="\t", index=False)

print(A_new_df)
print(B_new_df)

  index             x             v             a
0     x  1.000000e+00  1.100000e+00  2.886580e-15
1     v -1.281955e-16  9.000000e-01  9.500000e-01
2     a -9.313780e-17  5.551115e-16  1.100000e+00
  index             u
0     x -1.110223e-16
1     v -1.000000e-02
2     a  9.000000e-01


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [13]:
print("none")

none
